# Demo 03 — Why retrieval is hard: a question ≠ its answer

The thought-provoking one. A human reading *"why was my card charged twice?"* instantly connects it
to a policy clause about *"a duplicate authorization hold that's released within a few days"* —
**barely a word in common**. For a machine this is a research field. We'll watch the problem appear
and then fix it.

> Needs an embeddings endpoint (`EMBED_MODEL` in `.env`; e.g. Ollama `nomic-embed-text`). The chat
> model is reused for HyDE. This notebook is about *seeing the numbers*, not the framework.

In [1]:
import os
import numpy as np

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                        # reads ../.env  (no MLflow here — this demo is pure embeddings)

client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:11434/v1"),
                api_key=os.environ.get("OPENAI_API_KEY", "not-needed-for-local"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "nomic-embed-text")
CHAT_MODEL  = os.environ.get("MODEL", "qwen2.5:7b")

def embed(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("embeddings client ready:", EMBED_MODEL)


embeddings client ready: nomic-embed-text:latest


## The setup: one question, some other questions, and the real answer

Lexical (keyword) search fails immediately — the answer shares almost no words with the question.
Let's see what *semantic* embeddings do.

In [2]:
question = "Why was my card charged twice for one order?"

other_questions = [
    "How do I track my package?",
    "What is your refund window for sealed products?",
    "Why is there a second pending charge on my card?",   # a near-paraphrase of the question
]
the_answer = ("A duplicate charge is normally a temporary authorization hold from a retried payment; "
              "the pending amount drops off and is released back to your card within 3 to 5 business days.")

q = embed([question])[0]
print("Q vs OTHER QUESTIONS (symmetric: similar shape):")
for t, e in zip(other_questions, embed(other_questions)):
    print(f"  {cos(q, e):.3f}  {t}")
print(f"\nQ vs THE ACTUAL ANSWER:")
print(f"  {cos(q, embed([the_answer])[0]):.3f}  {the_answer[:60]}...")

Q vs OTHER QUESTIONS (symmetric: similar shape):
  0.535  How do I track my package?
  0.498  What is your refund window for sealed products?
  0.805  Why is there a second pending charge on my card?

Q vs THE ACTUAL ANSWER:
  0.699  A duplicate charge is normally a temporary authorization hol...


**The asymmetry.** Notice the question often sits *closer to other questions* than to the document
that actually answers it. Questions are short and interrogative; answers are long and declarative —
they live in **different regions of embedding space**. A general-purpose ("symmetric") model finds
look-alike *questions*, not *answers*. This is why naive "embed the query, grab nearest" disappoints.

Two production fixes:
1. **Asymmetric embedding models** trained on query→passage pairs (MS MARCO; E5 with `query:` /
   `passage:` prefixes; DPR dual encoders).
2. **HyDE** — turn the question into a *hypothetical answer*, then search with **that**. Now you're
   matching answer-to-answer.

## HyDE: search with a hypothetical answer

Ask the LLM to *write* the answer it expects, then search with **that** instead of the question. A
hypothetical answer shares the vocabulary, length, and shape of *real* answers, so it lands near
them. We generate a **few drafts and average** their embeddings — the real HyDE recipe — which
cancels out any single draft that guesses a detail wrong. Notice the power comes from the answer's
*shape*, not from the draft being factually correct.

In [3]:
# HyDE — Hypothetical Document Embeddings. Instead of searching with the QUESTION, ask the model to
# *write* the answer it expects and search with THAT. Generate several drafts and AVERAGE their
# embeddings (the paper uses ~8): averaging cancels out any single draft that guesses wrong.
hyde_prompt = ("In one sentence, explain the likely reason and what happens next, "
               f"as a store help-center answer to: {question}")
drafts = [
    client.chat.completions.create(
        model=CHAT_MODEL, temperature=0.7,
        messages=[{"role": "user", "content": hyde_prompt}],
    ).choices[0].message.content.strip()
    for _ in range(8)
]
print("One of the 8 hypothetical answers we generate and average:\n ", drafts[0], "\n")

a = embed([the_answer])[0]
h = embed(drafts).mean(axis=0)                 # the HyDE vector = mean of the drafts' embeddings
q_sim, h_sim = cos(q, a), cos(h, a)
verdict = "closer — the gap closed" if h_sim > q_sim else "no improvement on this run"
print(f"Question        -> answer similarity : {q_sim:.3f}")
print(f"HyDE (avg of 8) -> answer similarity : {h_sim:.3f}   ({verdict})")
print("\nWhy it works: a hypothetical answer is answer-SHAPED (declarative, policy vocabulary), so it")
print("lands near real answers — even if a single draft gets the yes/no details wrong.")

One of the 8 hypothetical answers we generate and average:
  Your card was charged twice for one order likely due to a processing error or duplicate transaction, which will be investigated, and you'll receive a credit or refund for theduplicate charge shortly. 

Question        -> answer similarity : 0.699
HyDE (avg of 8) -> answer similarity : 0.791   (closer — the gap closed)

Why it works: a hypothetical answer is answer-SHAPED (declarative, policy vocabulary), so it
lands near real answers — even if a single draft gets the yes/no details wrong.


## The takeaway

Keyword fails → embeddings fix vocabulary → but Q and A live apart → asymmetric models / HyDE bridge
the shape gap. And there's more in production: **hybrid search** (add BM25 so exact strings like
`SKU AZ-4471` aren't lost) and **reranking** (a cross-encoder re-scores the shortlist). The standard
pipeline is `hybrid → RRF fusion → cross-encoder rerank → top-k → LLM`.

The point of the whole session in one line: **all this machinery just to reproduce what your brain
does in half a second when you "go look it up."** Easy for a human; a research field for a machine.

*(RAG vs fine-tuning vs long-context — when to reach for which — is Session 8.)*